# DEMO MAP/Hessian Diagnostics

This notebook rebuilds the DEMO Stage 4 model from the persisted public artifacts, then runs `map_geometry_analysis` as a two-stage MAP/Hessian workflow:

- stride-30 multi-start MAP search to find a posterior basin cheaply.
- full-grid one-start polish from the coarse mode.
- full-grid likelihood/posterior Hessian curvature at the polished mode.

The Jacobian/SVD structural sensitivity diagnostic is separated into `demo_jacobian_diagnostics.ipynb`.


In [1]:
# Configuration
WORKSPACE_ID = "DEMO"
DIAGNOSTIC_SEED = 42

# Coarse basin search: cheap enough to afford multiple starts.
MAP_SEARCH_TIME_STRIDE = 30
MAP_SEARCH_N_STARTS = 8
MAP_SEARCH_PARALLEL_WORKERS = 4
MAP_SEARCH_OPTIMIZER_OPTIONS = {
    "maxiter": 80,
    "maxfun": 200,
    "ftol": 1e-6,
    "gtol": 1e-4,
    "maxls": 30,
}

# Full-grid polish: one start from the best coarse mode, then full-grid Hessian.
MAP_POLISH_TIME_STRIDE = 1
MAP_POLISH_OPTIMIZER_OPTIONS = {
    "maxiter": 80,
    "maxfun": 150,
    "ftol": 1e-7,
    "gtol": 1e-4,
    "maxls": 30,
}

run_config = {
    "workspace_id": WORKSPACE_ID,
    "diagnostic_seed": DIAGNOSTIC_SEED,
    "map_search_time_stride": MAP_SEARCH_TIME_STRIDE,
    "map_search_n_starts": MAP_SEARCH_N_STARTS,
    "map_search_parallel_workers": MAP_SEARCH_PARALLEL_WORKERS,
    "map_search_optimizer_options": MAP_SEARCH_OPTIMIZER_OPTIONS,
    "map_polish_time_stride": MAP_POLISH_TIME_STRIDE,
    "map_polish_optimizer_options": MAP_POLISH_OPTIMIZER_OPTIONS,
}


In [2]:
from __future__ import annotations

import json
import logging
import math
import sys
import time
import warnings
from collections import Counter
from dataclasses import asdict
from pathlib import Path
from typing import Any

import jax.numpy as jnp
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import polars as pl
from IPython.display import Markdown, display
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("causal_ssm_agent.models.ssm_compilation").setLevel(logging.ERROR)
pio.renderers.default = "plotly_mimetype"


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data/DEMO/run/stage-4.json").exists():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")


def summary_markdown(mapping: dict[str, Any], title: str) -> Markdown:
    lines = [f"**{title}**", "", "| Field | Value |", "| --- | --- |"]
    for key, value in mapping.items():
        if isinstance(value, (dict, list)):
            text = json.dumps(value, default=str)
        elif isinstance(value, float):
            text = f"{value:.6g}"
        else:
            text = str(value)
        if len(text) > 110:
            text = text[:107] + "..."
        lines.append(f"| `{key}` | {text} |")
    return Markdown("\n".join(lines))


REPO_ROOT = find_repo_root(Path.cwd().resolve())
DATA_PIPELINE = REPO_ROOT / "apps/data-pipeline"
SRC = DATA_PIPELINE / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from causal_ssm_agent.models.ssm.diagnostics import (
    get_stage4b_sweep_context,
    map_geometry_analysis,
)
from causal_ssm_agent.models.ssm_builder import prepare_model_runtime
from causal_ssm_agent.models.ssm_compiler import compile_ssm_artifact

run_dir = REPO_ROOT / "data" / WORKSPACE_ID / "run"
stage4 = json.loads((run_dir / "stage-4.json").read_text())
stage1b = json.loads((run_dir / "stage-1b.json").read_text())
data_for_model = pl.read_parquet(run_dir / "stage2-model-data.parquet")

t0 = time.time()
compiled_ssm = compile_ssm_artifact(
    stage4["model_spec"],
    stage4["authored_priors"],
    causal_spec=stage1b["causal_spec"],
)
runtime = prepare_model_runtime(data_for_model, compiled_ssm=compiled_ssm)
sweep_context = get_stage4b_sweep_context(runtime.model)

prep_summary = {
    "repo_root": str(REPO_ROOT),
    "run_dir": str(run_dir),
    "compile_prepare_seconds": round(time.time() - t0, 3),
    "n_rows_long": data_for_model.height,
    "observation_grid_shape": list(runtime.observations.shape),
    "time_grid_length": int(runtime.times.shape[0]),
    "n_manifest": int(runtime.spec.n_manifest),
    "n_latent": int(runtime.spec.n_latent),
    "flat_parameter_dim": int(sweep_context.flat_dim),
    "structural_backend": runtime.inference_structure.structural_backend,
    "resolved_method": runtime.inference_structure.resolved_method,
}

display(summary_markdown(run_config, "Run Configuration"))
display(summary_markdown(prep_summary, "Prepared Runtime"))


**Run Configuration**

| Field | Value |
| --- | --- |
| `workspace_id` | DEMO |
| `diagnostic_seed` | 42 |
| `map_search_time_stride` | 30 |
| `map_search_n_starts` | 8 |
| `map_search_parallel_workers` | 4 |
| `map_search_optimizer_options` | {"maxiter": 80, "maxfun": 200, "ftol": 1e-06, "gtol": 0.0001, "maxls": 30} |
| `map_polish_time_stride` | 1 |
| `map_polish_optimizer_options` | {"maxiter": 80, "maxfun": 150, "ftol": 1e-07, "gtol": 0.0001, "maxls": 30} |

**Prepared Runtime**

| Field | Value |
| --- | --- |
| `repo_root` | /Users/ma9o/Desktop/causal-ssm-agent/trees/main |
| `run_dir` | /Users/ma9o/Desktop/causal-ssm-agent/trees/main/data/DEMO/run |
| `compile_prepare_seconds` | 1.164 |
| `n_rows_long` | 27325 |
| `observation_grid_shape` | [1588, 23] |
| `time_grid_length` | 1588 |
| `n_manifest` | 23 |
| `n_latent` | 8 |
| `flat_parameter_dim` | 69 |
| `structural_backend` | composed |
| `resolved_method` | aux_gibbs |

In [3]:
STATUS_COLORS = {"pass": "#2f855a", "warn": "#d69e2e", "fail": "#c53030", "unknown": "#718096"}
FAMILY_TOKENS = ("lambda", "drift", "diffusion", "obs_r", "obs_shape", "manifest", "t0")
FAMILY_PALETTE = {
    "lambda": "#2b6cb0",
    "drift": "#805ad5",
    "diffusion": "#dd6b20",
    "obs_r": "#319795",
    "obs_shape": "#d69e2e",
    "manifest": "#9f7aea",
    "t0": "#38a169",
    "other": "#718096",
}


def _family_of(parameter: str, interpretable: str) -> str:
    text = f"{parameter} {interpretable}".lower()
    for token in FAMILY_TOKENS:
        if token in text:
            return token
    return "other"


def _status_value(entry: dict[str, Any]) -> str:
    for key in ("normalized_sv_status", "normalized_status", "sv_status", "status"):
        value = entry.get(key)
        if isinstance(value, str):
            return value
    return "unknown"


def _effective_value(entry: dict[str, Any]) -> float | None:
    value = entry.get("normalized_effective_sv", entry.get("normalized_effective_eigenvalue"))
    try:
        if value is None or not math.isfinite(float(value)):
            return None
        return float(value)
    except (TypeError, ValueError):
        return None


def _direction_lambda(direction: dict[str, Any]) -> float:
    raw = direction.get("normalized_eigenvalue", direction.get("normalized_singular_value"))
    try:
        value = float(raw)
    except (TypeError, ValueError):
        return float("nan")
    return value if math.isfinite(value) else float("nan")


def _finite_float(value: Any) -> float | None:
    try:
        out = float(value)
    except (TypeError, ValueError):
        return None
    return out if math.isfinite(out) else None


def _finite_matrix(values: list[list[float]] | np.ndarray, *, n_rows: int | None = None) -> np.ndarray | None:
    matrix = np.asarray(values, dtype=float)
    if matrix.ndim != 2:
        return None
    if n_rows is not None and matrix.shape[0] != n_rows:
        return None
    if not np.isfinite(matrix).any():
        return None
    return np.where(np.isfinite(matrix), matrix, 0.0)


def weak_parameter_rows(entries: list[dict[str, Any]]) -> list[dict[str, Any]]:
    rows = []
    for entry in entries:
        status = _status_value(entry)
        if status == "pass":
            continue
        rows.append({
            "parameter": entry.get("parameter"),
            "interpretable_parameter": entry.get("interpretable_parameter"),
            "status": status,
            "normalized_effective": _effective_value(entry),
            "raw_effective": entry.get("effective_sv", entry.get("effective_eigenvalue")),
        })
    return rows


def interpretable_lookup(per_parameter: list[dict[str, Any]]) -> dict[str, str]:
    return {
        str(entry["parameter"]): str(entry.get("interpretable_parameter") or entry["parameter"])
        for entry in per_parameter
    }


# ── 1. Family-grouped weak-direction loading heatmap ───────────────────────────
def family_grouped_loading_heatmap(
    *,
    parameter_names: list[str],
    interpretable_names: dict[str, str],
    full_vectors: list[list[float]] | np.ndarray,
    weak_direction_meta: list[dict[str, Any]],
    title: str,
    max_parameters: int = 40,
    loading_threshold: float = 0.05,
) -> go.Figure | None:
    if not weak_direction_meta or not parameter_names:
        return None
    matrix = _finite_matrix(full_vectors, n_rows=len(parameter_names))
    if matrix is None:
        return None

    direction_meta = [d for d in weak_direction_meta if 0 <= int(d["index"]) - 1 < matrix.shape[1]]
    if not direction_meta:
        return None
    direction_indices = [int(d["index"]) - 1 for d in direction_meta]
    sub = matrix[:, direction_indices]

    # When few directions, tighten cap so we don't draw a tall thin column
    effective_cap = min(max_parameters, 20 if len(direction_meta) <= 2 else max_parameters)
    abs_max = np.max(np.abs(sub), axis=1)
    keep = [i for i in range(len(parameter_names)) if abs_max[i] >= loading_threshold]
    if not keep:
        keep = list(np.argsort(abs_max)[::-1][: min(effective_cap, len(parameter_names))])
    if len(keep) > effective_cap:
        keep = sorted(keep, key=lambda i: -abs_max[i])[:effective_cap]

    annotated = []
    for i in keep:
        name = parameter_names[i]
        family = _family_of(name, interpretable_names.get(name, ""))
        family_rank = FAMILY_TOKENS.index(family) if family in FAMILY_TOKENS else len(FAMILY_TOKENS)
        annotated.append((family_rank, family, name, i))
    annotated.sort(key=lambda row: (row[0], row[2]))
    ordered_indices = [row[3] for row in annotated]
    families_in_order = [row[1] for row in annotated]
    y_labels = [interpretable_names.get(parameter_names[i], parameter_names[i]) for i in ordered_indices]

    z = sub[ordered_indices, :]
    abs_z_max = max(float(np.max(np.abs(z))), 1e-6) if z.size else 1.0

    x_labels = [f"D{d['index']} · {d.get('status', '?')}" for d in direction_meta]

    hover_text = []
    for row_idx, param_idx in enumerate(ordered_indices):
        row_hover = []
        for col_idx, d in enumerate(direction_meta):
            lam = _direction_lambda(d)
            label = interpretable_names.get(parameter_names[param_idx], parameter_names[param_idx])
            row_hover.append(
                f"<b>{label}</b>"
                f"<br>family={families_in_order[row_idx]}"
                f"<br>direction D{d['index']} ({d.get('status', '?')})"
                f"<br>normalized λ={lam:.3g}"
                f"<br>loading={z[row_idx, col_idx]:+.4f}"
            )
        hover_text.append(row_hover)

    fig = go.Figure(
        go.Heatmap(
            z=z,
            x=x_labels,
            y=y_labels,
            colorscale="RdBu",
            zmin=-abs_z_max,
            zmax=abs_z_max,
            zmid=0,
            text=hover_text,
            hovertemplate="%{text}<extra></extra>",
            colorbar={"title": "signed loading", "thickness": 14},
            xgap=1,
            ygap=1,
        )
    )

    shapes = []
    annotations = []
    last_family = None
    block_start = -0.5
    for row_idx, family in enumerate(families_in_order + [None]):
        if family != last_family and last_family is not None:
            shapes.append(
                {
                    "type": "line",
                    "xref": "paper",
                    "x0": 0,
                    "x1": 1,
                    "yref": "y",
                    "y0": row_idx - 0.5,
                    "y1": row_idx - 0.5,
                    "line": {"color": "#1a202c", "width": 1, "dash": "dot"},
                }
            )
            annotations.append(
                {
                    "xref": "paper",
                    "yref": "y",
                    "x": 1.02,
                    "xanchor": "left",
                    "y": (block_start + row_idx - 0.5) / 2,
                    "yanchor": "middle",
                    "text": f"<b>{last_family}</b>",
                    "showarrow": False,
                    "font": {"color": FAMILY_PALETTE.get(last_family, "#718096"), "size": 11},
                }
            )
            block_start = row_idx - 0.5
        if family != last_family:
            last_family = family

    height = min(900, max(360, 22 * len(y_labels) + 140))
    fig.update_layout(
        title=title,
        xaxis_title="Weak direction",
        yaxis_title="Parameter (grouped by family)",
        template="plotly_white",
        height=height,
        margin={"l": 320, "r": 110, "t": 70, "b": 60},
        shapes=shapes,
        annotations=annotations,
    )
    return fig


# ── 2. Laplace 2D ellipses for top weak-direction parameter pairs ──────────────
def laplace_pair_ellipses(
    *,
    eigvecs_norm: list[list[float]] | np.ndarray,
    eigvals_norm: list[float] | np.ndarray,
    parameter_names: list[str],
    interpretable_names: dict[str, str],
    weak_directions: list[dict[str, Any]],
    title: str,
    max_pairs: int = 6,
) -> go.Figure | None:
    if not weak_directions or not parameter_names:
        return None
    V = _finite_matrix(eigvecs_norm, n_rows=len(parameter_names))
    lam = np.asarray(eigvals_norm, dtype=float)
    if V is None or lam.ndim != 1 or lam.size != V.shape[1]:
        return None

    finite_positive = np.isfinite(lam) & (lam > 1e-8)
    if not np.any(finite_positive):
        return None
    H_inv_norm = (V[:, finite_positive] / lam[finite_positive][None, :]) @ V[:, finite_positive].T

    name_to_idx = {name: i for i, name in enumerate(parameter_names)}
    pairs: list[tuple[int, int, dict[str, Any]]] = []
    seen: set[tuple[int, int]] = set()
    for direction in weak_directions:
        direction_index = int(direction.get("index", 0)) - 1
        if direction_index < 0 or direction_index >= lam.size or not finite_positive[direction_index]:
            continue
        loadings = direction.get("top_loadings", [])
        if len(loadings) < 2:
            continue
        i = name_to_idx.get(str(loadings[0].get("parameter")))
        j = name_to_idx.get(str(loadings[1].get("parameter")))
        if i is None or j is None or i == j:
            continue
        key = (min(i, j), max(i, j))
        if key in seen:
            continue
        seen.add(key)
        pairs.append((i, j, direction))
        if len(pairs) >= max_pairs:
            break
    if not pairs:
        return None

    cols = min(3, len(pairs))
    rows_n = math.ceil(len(pairs) / cols)
    subplot_titles = []
    for i, j, direction in pairs:
        eig_value = _direction_lambda(direction)
        subplot_titles.append(
            f"D{direction['index']} (λ_norm={eig_value:.3g})"
            f"<br><sub>{interpretable_names.get(parameter_names[i], parameter_names[i])} ↔ "
            f"{interpretable_names.get(parameter_names[j], parameter_names[j])}</sub>"
        )

    fig = make_subplots(
        rows=rows_n,
        cols=cols,
        subplot_titles=subplot_titles,
        horizontal_spacing=0.10,
        vertical_spacing=0.18,
    )
    theta = np.linspace(0.0, 2.0 * np.pi, 100)
    unit_circle = np.stack([np.cos(theta), np.sin(theta)])
    legend_added = False

    for k, (i, j, direction) in enumerate(pairs):
        row = k // cols + 1
        col = k % cols + 1
        sub_cov = H_inv_norm[np.ix_([i, j], [i, j])]
        sub_cov = 0.5 * (sub_cov + sub_cov.T)
        if not np.all(np.isfinite(sub_cov)):
            continue
        cov_eigvals, cov_eigvecs = np.linalg.eigh(sub_cov)
        cov_eigvals = np.maximum(cov_eigvals, 0.0)
        max_radius = float(np.sqrt(np.max(cov_eigvals))) * 2.0 if cov_eigvals.size else 1.0
        if not math.isfinite(max_radius):
            continue
        max_radius = max(max_radius, 1e-3)
        for sigma, dash in [(1, "solid"), (2, "dash")]:
            radii = np.sqrt(cov_eigvals) * sigma
            ellipse = cov_eigvecs @ (radii[:, None] * unit_circle)
            if not np.all(np.isfinite(ellipse)):
                continue
            fig.add_trace(
                go.Scatter(
                    x=ellipse[0],
                    y=ellipse[1],
                    mode="lines",
                    line={"color": "#805ad5", "width": 1.6, "dash": dash},
                    name=f"{sigma}σ Laplace",
                    legendgroup=f"sigma{sigma}",
                    showlegend=not legend_added,
                    hovertemplate=f"{sigma}σ Laplace<extra></extra>",
                ),
                row=row,
                col=col,
            )
        idx_dir = int(direction["index"]) - 1
        if 0 <= idx_dir < V.shape[1]:
            v_full = V[:, idx_dir]
            weak_dir_axis = np.array([v_full[i], v_full[j]])
            norm = float(np.linalg.norm(weak_dir_axis))
            if math.isfinite(norm) and norm > 1e-9:
                arrow = (weak_dir_axis / norm) * max_radius
                fig.add_trace(
                    go.Scatter(
                        x=[-arrow[0], arrow[0]],
                        y=[-arrow[1], arrow[1]],
                        mode="lines",
                        line={"color": "#dd6b20", "width": 1.4, "dash": "dot"},
                        name="weak direction axis",
                        legendgroup="weakdir",
                        showlegend=not legend_added,
                        hovertemplate="weak direction axis<extra></extra>",
                    ),
                    row=row,
                    col=col,
                )
        fig.add_trace(
            go.Scatter(
                x=[0],
                y=[0],
                mode="markers",
                marker={"size": 9, "color": "#c53030", "symbol": "x"},
                name="MAP",
                legendgroup="map",
                showlegend=not legend_added,
                hovertemplate="MAP<extra></extra>",
            ),
            row=row,
            col=col,
        )
        legend_added = True
        axis_range = [-max_radius * 1.15, max_radius * 1.15]
        fig.update_xaxes(
            title_text=f"<sub>{parameter_names[i]}</sub>",
            range=axis_range,
            row=row,
            col=col,
            zeroline=True,
            zerolinecolor="#cbd5e0",
            constrain="domain",
        )
        fig.update_yaxes(
            title_text=f"<sub>{parameter_names[j]}</sub>",
            range=axis_range,
            row=row,
            col=col,
            zeroline=True,
            zerolinecolor="#cbd5e0",
            scaleanchor=f"x{(k + 1) if k > 0 else ''}",
            scaleratio=1,
            constrain="domain",
        )

    if not fig.data:
        return None
    fig.update_layout(
        title=f"{title} — Laplace ellipses on (z−z*)/σ_prior",
        template="plotly_white",
        height=380 * rows_n + 100,
        margin={"l": 70, "r": 30, "t": 110, "b": 60},
        legend={"orientation": "h", "yanchor": "bottom", "y": -0.12, "xanchor": "center", "x": 0.5},
    )
    return fig


# ── 3. Profile log-likelihood along each weak Hessian direction ────────────────
def profile_along_directions(
    *,
    log_lik_fn,
    observations: jnp.ndarray,
    times: jnp.ndarray,
    z_map: list[float] | np.ndarray,
    prior_std: list[float] | np.ndarray,
    eigvecs_norm: list[list[float]] | np.ndarray,
    weak_directions: list[dict[str, Any]],
    title: str,
    max_directions: int = 6,
    n_grid: int = 21,
    alpha_max: float = 3.0,
) -> go.Figure | None:
    if not weak_directions:
        return None
    z = np.asarray(z_map, dtype=np.float64)
    sigma = np.asarray(prior_std, dtype=np.float64)
    V = _finite_matrix(eigvecs_norm, n_rows=z.size)
    if z.size == 0 or sigma.size != z.size or V is None:
        return None
    if not np.all(np.isfinite(z)) or not np.all(np.isfinite(sigma)):
        return None

    valid_directions = []
    for direction in weak_directions:
        idx = int(direction.get("index", 0)) - 1
        eig_val = _direction_lambda(direction)
        if 0 <= idx < V.shape[1] and math.isfinite(eig_val):
            valid_directions.append(direction)
        if len(valid_directions) >= max_directions:
            break
    if not valid_directions:
        return None

    log_lik_at_map = _finite_float(log_lik_fn(jnp.asarray(z), observations, times))
    if log_lik_at_map is None:
        return None

    cols = min(3, len(valid_directions))
    rows_n = math.ceil(len(valid_directions) / cols)
    subplot_titles = [
        f"D{d['index']} (λ_norm={_direction_lambda(d):.3g}, {d.get('status', '?')})"
        for d in valid_directions
    ]
    fig = make_subplots(
        rows=rows_n,
        cols=cols,
        subplot_titles=subplot_titles,
        horizontal_spacing=0.08,
        vertical_spacing=0.20,
    )
    alphas = np.linspace(-alpha_max, alpha_max, n_grid)

    legend_added = False
    for k, direction in enumerate(valid_directions):
        idx = int(direction["index"]) - 1
        v_norm = V[:, idx]
        scaled = sigma * v_norm
        if not np.all(np.isfinite(scaled)):
            continue
        actual = []
        for alpha in alphas:
            z_step = jnp.asarray(z + alpha * scaled, dtype=jnp.float64)
            actual.append(_finite_float(log_lik_fn(z_step, observations, times)))
        if all(value is None for value in actual):
            continue
        eig_val = _direction_lambda(direction)
        parabola = log_lik_at_map - 0.5 * eig_val * alphas**2

        row = k // cols + 1
        col = k % cols + 1
        fig.add_trace(
            go.Scatter(
                x=alphas,
                y=actual,
                mode="lines+markers",
                line={"color": "#2b6cb0", "width": 2},
                marker={"size": 5},
                name="actual log-lik profile",
                legendgroup="actual",
                showlegend=not legend_added,
                hovertemplate="α=%{x:.2f}<br>log L=%{y:.4g}<extra></extra>",
            ),
            row=row,
            col=col,
        )
        fig.add_trace(
            go.Scatter(
                x=alphas,
                y=parabola,
                mode="lines",
                line={"color": "#dd6b20", "width": 1.6, "dash": "dash"},
                name="Hessian-implied parabola",
                legendgroup="hessian",
                showlegend=not legend_added,
                hovertemplate="α=%{x:.2f}<br>parabola=%{y:.4g}<extra></extra>",
            ),
            row=row,
            col=col,
        )
        legend_added = True
        fig.update_xaxes(title_text="α", row=row, col=col)
        if col == 1:
            fig.update_yaxes(title_text="log likelihood", row=row, col=col)

    if not fig.data:
        return None
    fig.update_layout(
        title=f"{title} — actual log-lik (blue) vs Hessian quadratic (orange) along weak directions",
        template="plotly_white",
        height=300 * rows_n + 80,
        margin={"l": 70, "r": 24, "t": 90, "b": 70},
        legend={"orientation": "h", "yanchor": "bottom", "y": -0.18, "xanchor": "center", "x": 0.5},
    )
    return fig


# ── 4. Diverging-color eigenvalue strip ────────────────────────────────────────
def eigenvalue_strip_figure(map_payload: dict[str, Any]) -> go.Figure | None:
    fig = make_subplots(
        rows=2,
        cols=1,
        subplot_titles=["Likelihood Hessian (normalized)", "Posterior Hessian (normalized)"],
        vertical_spacing=0.40,
    )
    bounds = []
    signed_by_key: dict[str, np.ndarray] = {}
    eigvals_by_key: dict[str, np.ndarray] = {}
    for key in ("likelihood_curvature", "posterior_curvature"):
        eigvals = np.asarray(map_payload[key]["normalized_eigenvalues"], dtype=float)
        eigvals_by_key[key] = eigvals
        signed = np.full_like(eigvals, np.nan, dtype=float)
        finite = np.isfinite(eigvals)
        signed[finite] = np.sign(eigvals[finite]) * np.log10(np.abs(eigvals[finite]) + 1e-8)
        signed_by_key[key] = signed
        finite_signed = signed[np.isfinite(signed)]
        if finite_signed.size:
            bounds.append(float(np.max(np.abs(finite_signed))))
    color_max = max(bounds) if bounds else 1.0
    if not math.isfinite(color_max) or color_max <= 0:
        color_max = 1.0

    for row, (key, label) in enumerate([
        ("likelihood_curvature", "H_lik"),
        ("posterior_curvature", "H_post"),
    ], start=1):
        eigvals = eigvals_by_key[key]
        if eigvals.size == 0:
            continue
        signed = signed_by_key[key]
        z = signed[None, :]
        text = [
            [
                (
                    f"rank={i + 1}<br>λ_norm={float(val):.4g}<br>signed log10|λ|={float(s):.3f}"
                    if math.isfinite(float(val)) and math.isfinite(float(s))
                    else f"rank={i + 1}<br>λ_norm=non-finite"
                )
                for i, (val, s) in enumerate(zip(eigvals, signed, strict=True))
            ]
        ]
        fig.add_trace(
            go.Heatmap(
                z=z,
                x=np.arange(1, eigvals.size + 1),
                y=[label],
                colorscale="RdBu_r",
                zmid=0,
                zmin=-color_max,
                zmax=color_max,
                text=text,
                hovertemplate="%{text}<extra></extra>",
                showscale=(row == 1),
                colorbar={"title": "signed log10|λ_norm|", "thickness": 14, "len": 0.9} if row == 1 else None,
                xgap=1,
            ),
            row=row,
            col=1,
        )
        fig.update_xaxes(title_text="Eigenvalue rank", row=row, col=1)

    if not fig.data:
        return None
    fig.update_layout(
        title="Hessian eigenvalue strip — red = negative, blue = positive, white ≈ 1",
        template="plotly_white",
        height=320,
        margin={"l": 90, "r": 90, "t": 90, "b": 60},
    )
    return fig


# ── 5. MAP starts: parcoords for many starts, bar fallback for one ─────────────
def starts_visual(map_payload: dict[str, Any]) -> go.Figure | None:
    starts = map_payload.get("starts") or []
    if not starts:
        return None
    if len(starts) >= 2:
        return _starts_parallel_coords(starts)
    return _single_start_bar(starts[0])


def _starts_parallel_coords(starts: list[dict[str, Any]]) -> go.Figure:
    indices = [s["index"] for s in starts]
    start_lp = [_finite_float(s.get("start_log_posterior")) or 0.0 for s in starts]
    final_lp = [_finite_float(s.get("log_posterior")) or 0.0 for s in starts]
    log_lik = [_finite_float(s.get("log_likelihood")) or 0.0 for s in starts]
    log_prior = [_finite_float(s.get("log_prior")) or 0.0 for s in starts]
    log_grad = [math.log10(max(_finite_float(s.get("grad_norm")) or 0.0, 1e-12)) for s in starts]
    distance = [_finite_float(s.get("distance_to_best")) or 0.0 for s in starts]
    success = [1 if s.get("success") else 0 for s in starts]
    start_kinds = [s.get("start_kind", "?") for s in starts]
    kind_codes = {kind: idx for idx, kind in enumerate(sorted(set(start_kinds)))}
    kind_values = [kind_codes[k] for k in start_kinds]

    dimensions = [
        {"label": "start index", "values": indices},
        {
            "label": "start kind",
            "values": kind_values,
            "tickvals": list(kind_codes.values()),
            "ticktext": list(kind_codes.keys()),
        },
        {"label": "start log post", "values": start_lp},
        {"label": "final log post", "values": final_lp},
        {"label": "final log lik", "values": log_lik},
        {"label": "final log prior", "values": log_prior},
        {"label": "log10 grad norm", "values": log_grad},
        {"label": "dist to best", "values": distance},
    ]
    fig = go.Figure(
        go.Parcoords(
            line={
                "color": success,
                "colorscale": [[0.0, "#c53030"], [1.0, "#2f855a"]],
                "showscale": False,
                "cmin": 0,
                "cmax": 1,
            },
            dimensions=dimensions,
        )
    )
    fig.update_layout(
        title="MAP starts (parallel coordinates) — green = success, red = failure",
        template="plotly_white",
        height=420,
        margin={"l": 90, "r": 90, "t": 90, "b": 60},
    )
    return fig


def _single_start_bar(start: dict[str, Any]) -> go.Figure:
    metrics = {
        "start log posterior": float(start.get("start_log_posterior") or 0.0),
        "final log posterior": float(start.get("log_posterior") or 0.0),
        "final log likelihood": float(start.get("log_likelihood") or 0.0),
        "final log prior": float(start.get("log_prior") or 0.0),
        "log10 grad norm": math.log10(max(float(start.get("grad_norm") or 0.0), 1e-12)),
    }
    labels = list(metrics)
    values = [metrics[k] for k in labels]
    success = bool(start.get("success"))
    color = "#2f855a" if success else "#c53030"
    title = (
        f"Single MAP start (kind={start.get('start_kind')}, success={success}, "
        f"iters={start.get('n_iters')})"
    )
    fig = go.Figure(
        go.Bar(
            x=values,
            y=labels,
            orientation="h",
            marker_color=color,
            hovertemplate="%{y}=%{x:.4g}<extra></extra>",
        )
    )
    fig.update_layout(
        title=title,
        template="plotly_white",
        height=260,
        xaxis_title="value (log scale where labelled)",
        margin={"l": 180, "r": 30, "t": 70, "b": 40},
    )
    return fig


# Notebook alias used below
starts_parallel_coords = starts_visual


# ── 6. Bipartite weak-direction ↔ parameter graph ──────────────────────────────
def bipartite_loading_graph(
    *,
    parameter_names: list[str],
    interpretable_names: dict[str, str],
    full_vectors: list[list[float]] | np.ndarray,
    weak_direction_meta: list[dict[str, Any]],
    title: str,
    max_parameters: int = 24,
    loading_threshold: float = 0.15,
    min_directions: int = 2,
) -> go.Figure | None:
    if not weak_direction_meta or not parameter_names:
        return None
    # Bipartite is only informative once there are multiple directions to compare
    if len(weak_direction_meta) < min_directions:
        return None
    matrix = np.asarray(full_vectors, dtype=float)
    direction_meta = [d for d in weak_direction_meta if 0 <= int(d["index"]) - 1 < matrix.shape[1]]
    if len(direction_meta) < min_directions:
        return None
    direction_indices = [int(d["index"]) - 1 for d in direction_meta]
    sub = matrix[:, direction_indices]
    abs_max = np.max(np.abs(sub), axis=1)
    keep = [i for i in range(len(parameter_names)) if abs_max[i] >= loading_threshold]
    if len(keep) < 2:
        return None
    keep = sorted(keep, key=lambda i: -abs_max[i])[:max_parameters]
    keep.sort(
        key=lambda i: (
            FAMILY_TOKENS.index(_family_of(parameter_names[i], interpretable_names.get(parameter_names[i], "")))
            if _family_of(parameter_names[i], interpretable_names.get(parameter_names[i], "")) in FAMILY_TOKENS
            else len(FAMILY_TOKENS),
            parameter_names[i],
        )
    )

    n_dirs = len(direction_meta)
    n_params = len(keep)
    dir_x = np.linspace(-1.0, 1.0, n_dirs) if n_dirs > 1 else np.array([0.0])
    param_x = np.linspace(-1.0, 1.0, n_params) if n_params > 1 else np.array([0.0])

    edge_traces = []
    for col_idx, direction in enumerate(direction_meta):
        for row_local, param_idx in enumerate(keep):
            value = float(sub[param_idx, col_idx])
            if abs(value) < loading_threshold:
                continue
            color = "#2b6cb0" if value > 0 else "#c53030"
            width = 0.5 + 4.0 * abs(value)
            edge_traces.append(
                go.Scatter(
                    x=[dir_x[col_idx], param_x[row_local]],
                    y=[1.0, 0.0],
                    mode="lines",
                    line={"color": color, "width": width},
                    opacity=min(1.0, 0.25 + abs(value)),
                    hovertemplate=(
                        f"D{direction['index']} ({direction.get('status', '?')})"
                        f" ↔ {interpretable_names.get(parameter_names[param_idx], parameter_names[param_idx])}"
                        f"<br>loading={value:+.3f}<extra></extra>"
                    ),
                    showlegend=False,
                )
            )

    direction_node = go.Scatter(
        x=dir_x,
        y=[1.0] * n_dirs,
        mode="markers+text",
        marker={
            "size": 22,
            "color": [STATUS_COLORS.get(d.get("status", "unknown"), STATUS_COLORS["unknown"]) for d in direction_meta],
            "line": {"color": "#1a202c", "width": 1},
        },
        text=[f"D{d['index']}" for d in direction_meta],
        textposition="top center",
        hovertemplate=[
            f"D{d['index']}<br>status={d.get('status', '?')}<br>λ_norm={_direction_lambda(d):.3g}<extra></extra>"
            for d in direction_meta
        ],
        showlegend=False,
    )

    param_colors = []
    param_text = []
    for i in keep:
        name = parameter_names[i]
        family = _family_of(name, interpretable_names.get(name, ""))
        param_colors.append(FAMILY_PALETTE.get(family, FAMILY_PALETTE["other"]))
        param_text.append(interpretable_names.get(name, name))

    param_node = go.Scatter(
        x=param_x,
        y=[0.0] * n_params,
        mode="markers+text",
        marker={"size": 16, "color": param_colors, "line": {"color": "#1a202c", "width": 1}},
        text=param_text,
        textposition="bottom center",
        hovertemplate=[
            f"<b>{interpretable_names.get(parameter_names[i], parameter_names[i])}</b><br>"
            f"{parameter_names[i]}<br>"
            f"family={_family_of(parameter_names[i], interpretable_names.get(parameter_names[i], ''))}<br>"
            f"max |loading|={abs_max[i]:.3f}<extra></extra>"
            for i in keep
        ],
        showlegend=False,
    )

    fig = go.Figure(data=edge_traces + [direction_node, param_node])
    fig.update_layout(
        title=f"{title} — edge thickness = |loading|, blue = +, red = −",
        template="plotly_white",
        height=480,
        margin={"l": 30, "r": 30, "t": 80, "b": 110},
        xaxis={"visible": False, "range": [-1.2, 1.2]},
        yaxis={"visible": False, "range": [-0.4, 1.4]},
    )
    return fig


# ── 7. Weak-parameter family aggregate ─────────────────────────────────────────
def family_grouped_bar(rows: list[dict[str, Any]], title: str) -> go.Figure | None:
    if not rows:
        return None
    counts: dict[tuple[str, str], int] = {}
    for row in rows:
        counts[(row["source"], row["family"])] = counts.get((row["source"], row["family"]), 0) + 1
    sources = sorted({source for source, _ in counts})
    families = [family for family in FAMILY_TOKENS if any(family == fam for _, fam in counts)]
    if not families:
        families = sorted({fam for _, fam in counts})

    fig = go.Figure()
    for source in sources:
        fig.add_trace(
            go.Bar(
                x=families,
                y=[counts.get((source, fam), 0) for fam in families],
                name=source,
                marker_color=[FAMILY_PALETTE.get(fam, FAMILY_PALETTE["other"]) for fam in families],
                marker_line={"color": "#1a202c", "width": 1},
                hovertemplate=f"source={source}<br>family=%{{x}}<br>count=%{{y}}<extra></extra>",
                opacity=0.92,
            )
        )
    fig.update_layout(
        title=title,
        barmode="group",
        template="plotly_white",
        xaxis_title="parameter family",
        yaxis_title="weak-parameter count",
        height=420,
        margin={"l": 60, "r": 30, "t": 70, "b": 60},
        legend={"orientation": "h", "yanchor": "bottom", "y": -0.22, "xanchor": "center", "x": 0.5},
    )
    return fig


# Backwards-compat alias for any prior callers
family_treemap = family_grouped_bar


# ── Existing supporting helpers ────────────────────────────────────────────────
def weak_parameter_bar(rows: list[dict[str, Any]], title: str, limit: int = 30) -> go.Figure | None:
    usable = [row for row in rows if row.get("normalized_effective") is not None]
    usable = sorted(usable, key=lambda row: float(row["normalized_effective"]))[:limit]
    if not usable:
        return None
    labels = [row["interpretable_parameter"] or row["parameter"] for row in usable]
    values = [max(abs(float(row["normalized_effective"])), 1e-12) for row in usable]
    colors = [STATUS_COLORS.get(row["status"], STATUS_COLORS["unknown"]) for row in usable]
    hover = [
        f"parameter={row['parameter']}<br>status={row['status']}<br>"
        f"normalized={row['normalized_effective']:.4g}<br>raw={row['raw_effective']}"
        for row in usable
    ]
    fig = go.Figure(
        go.Bar(
            x=values,
            y=labels,
            orientation="h",
            marker_color=colors,
            customdata=hover,
            hovertemplate="%{customdata}<extra></extra>",
        )
    )
    fig.add_vline(x=1, line_dash="dash", line_color="#c53030", annotation_text="fail")
    fig.add_vline(x=10, line_dash="dash", line_color="#2f855a", annotation_text="pass")
    height = max(220, 28 * len(usable) + 100)
    fig.update_layout(
        title=title,
        xaxis_title="Normalized effective value (log scale)",
        xaxis_type="log",
        template="plotly_white",
        height=height,
        margin={"l": 290, "r": 24, "t": 58, "b": 48},
    )
    return fig


def spectrum_figure(values: list[float], title: str, y_title: str) -> go.Figure:
    xs = list(range(1, len(values) + 1))
    ys = [max(float(v), 1e-12) for v in values]
    fig = go.Figure(
        go.Scatter(
            x=xs,
            y=ys,
            mode="lines+markers",
            line={"color": "#2b6cb0", "width": 2},
            marker={"size": 5},
            hovertemplate="rank=%{x}<br>value=%{y:.4g}<extra></extra>",
        )
    )
    fig.add_hline(y=1, line_dash="dash", line_color="#c53030", annotation_text="fail threshold")
    fig.add_hline(y=10, line_dash="dash", line_color="#2f855a", annotation_text="pass threshold")
    fig.update_layout(
        title=title,
        xaxis_title="Spectrum rank",
        yaxis_title=y_title,
        yaxis_type="log",
        template="plotly_white",
        height=400,
        margin={"l": 60, "r": 24, "t": 58, "b": 48},
    )
    return fig


def family_hit_rows(entries_by_source: dict[str, list[dict[str, Any]]]) -> list[dict[str, Any]]:
    rows = []
    for source, entries in entries_by_source.items():
        for entry in entries:
            status = _status_value(entry)
            if status == "pass":
                continue
            name = str(entry.get("parameter", ""))
            display_name = str(entry.get("interpretable_parameter", ""))
            family = _family_of(name, display_name)
            rows.append({
                "source": source,
                "family": family,
                "parameter": name,
                "interpretable_parameter": display_name,
                "status": status,
                "normalized_effective": _effective_value(entry),
            })
    return rows


def summary_dashboard_figure(sensitivity_summary: dict[str, Any], map_summary: dict[str, Any]) -> go.Figure:
    labels = [
        "Jacobian deficient dirs",
        "Jacobian weak dirs",
        "Jacobian weak params",
        "H_lik deficient dirs",
        "H_lik negative dirs",
        "H_post deficient dirs",
        "Boundary params",
    ]
    values = [
        sensitivity_summary["deficiency_count"],
        sensitivity_summary["weak_direction_count"],
        sensitivity_summary["weak_parameter_count"],
        map_summary.get("likelihood_deficiency_count", 0),
        map_summary.get("likelihood_negative_direction_count", 0),
        map_summary.get("posterior_deficiency_count", 0),
        len(map_summary.get("boundary_parameters") or []),
    ]
    colors = ["#c53030" if value else "#2f855a" for value in values]
    fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors, hovertemplate="%{x}<br>count=%{y}<extra></extra>"))
    fig.update_layout(
        title="Diagnostic Issue Counts",
        xaxis_title="",
        yaxis_title="Count",
        template="plotly_white",
        height=380,
        margin={"l": 60, "r": 24, "t": 58, "b": 110},
    )
    fig.update_xaxes(tickangle=-25)
    return fig


## Two-Stage MAP/Hessian Geometry

The coarse search uses every 30th time point only to find a good basin. The final reported MAP/Hessian summary comes from a full-grid polish and full-grid curvature calculation.


In [ ]:
map_search_obs = runtime.observations[::MAP_SEARCH_TIME_STRIDE]
map_search_times = runtime.times[::MAP_SEARCH_TIME_STRIDE]
map_obs = runtime.observations[::MAP_POLISH_TIME_STRIDE]
map_times = runtime.times[::MAP_POLISH_TIME_STRIDE]


def _has_finite_curvature_arrays(payload: dict[str, Any]) -> bool:
    for family in ("likelihood_curvature", "posterior_curvature"):
        curvature = payload.get(family) or {}
        eigvals = np.asarray(curvature.get("normalized_eigenvalues") or [], dtype=float)
        eigvecs = np.asarray(curvature.get("eigenvectors_normalized") or [], dtype=float)
        if eigvals.size == 0 or eigvecs.size == 0:
            return False
        if not np.all(np.isfinite(eigvals)) or not np.all(np.isfinite(eigvecs)):
            return False
    return True


search_t0 = time.time()
try:
    coarse_map_geometry = map_geometry_analysis(
        runtime.model,
        map_search_obs,
        map_search_times,
        n_starts=MAP_SEARCH_N_STARTS,
        seed=DIAGNOSTIC_SEED,
        sweep_context=sweep_context,
        optimizer_options=MAP_SEARCH_OPTIMIZER_OPTIONS,
        parallel_workers=MAP_SEARCH_PARALLEL_WORKERS,
    )
    coarse_map_payload = asdict(coarse_map_geometry)
    coarse_map_error = None
except Exception as exc:  # keep failure persisted in notebook state
    coarse_map_geometry = None
    coarse_map_payload = None
    coarse_map_error = {"type": type(exc).__name__, "message": str(exc)}

search_summary = {
    "elapsed_seconds": round(time.time() - search_t0, 3),
    "grid_stride": MAP_SEARCH_TIME_STRIDE,
    "grid_shape": list(map_search_obs.shape),
    "n_starts": MAP_SEARCH_N_STARTS,
    "parallel_workers": MAP_SEARCH_PARALLEL_WORKERS,
    "optimizer_options": MAP_SEARCH_OPTIMIZER_OPTIONS,
    "error": coarse_map_error,
}
if coarse_map_payload is not None:
    search_summary.update({
        "n_successful_starts": coarse_map_payload["n_successful_starts"],
        "best_start_index": coarse_map_payload["best_start_index"],
        "map_log_posterior": coarse_map_payload["map_log_posterior"],
        "map_log_likelihood": coarse_map_payload["map_log_likelihood"],
        "map_log_prior": coarse_map_payload["map_log_prior"],
        "final_grad_norm": coarse_map_payload["final_grad_norm"],
        "runner_up_objective_gap": coarse_map_payload["runner_up_objective_gap"],
        "boundary_parameters": coarse_map_payload["boundary_parameters"],
    })

display(summary_markdown(search_summary, "Stride-30 MAP Search Summary"))
if coarse_map_payload is not None:
    coarse_starts_fig = starts_parallel_coords(coarse_map_payload)
    if coarse_starts_fig is not None:
        display(coarse_starts_fig)

polish_t0 = time.time()
if coarse_map_payload is None or not coarse_map_payload.get("z_map_unconstrained"):
    map_geometry = None
    map_payload = None
    map_error = {"type": "Skipped", "message": "coarse MAP search did not produce a usable mode"}
else:
    try:
        map_geometry = map_geometry_analysis(
            runtime.model,
            map_obs,
            map_times,
            n_starts=1,
            seed=DIAGNOSTIC_SEED,
            sweep_context=sweep_context,
            optimizer_options=MAP_POLISH_OPTIMIZER_OPTIONS,
            parallel_workers=1,
            initial_starts=[coarse_map_payload["z_map_unconstrained"]],
            initial_start_kinds=[f"stride_{MAP_SEARCH_TIME_STRIDE}_map"],
        )
        map_payload = asdict(map_geometry)
        map_error = None
    except Exception as exc:  # keep failure persisted in notebook state
        map_geometry = None
        map_payload = None
        map_error = {"type": type(exc).__name__, "message": str(exc)}

map_curvature_plots_enabled = False
map_curvature_plot_warning = None
if map_payload is not None:
    if map_payload["n_successful_starts"] <= 0:
        map_curvature_plot_warning = "Full-grid MAP polish returned no successful starts; curvature plots skipped."
    elif not math.isfinite(float(map_payload["final_grad_norm"])):
        map_curvature_plot_warning = "Full-grid MAP polish returned a non-finite gradient norm; curvature plots skipped."
    elif not _has_finite_curvature_arrays(map_payload):
        map_curvature_plot_warning = "Full-grid Hessian output contains non-finite arrays; curvature plots skipped."
    else:
        map_curvature_plots_enabled = True

map_summary = {
    "elapsed_seconds": round(time.time() - polish_t0, 3),
    "grid_stride": MAP_POLISH_TIME_STRIDE,
    "grid_shape": list(map_obs.shape),
    "n_starts": 1,
    "parallel_workers": 1,
    "optimizer_options": MAP_POLISH_OPTIMIZER_OPTIONS,
    "error": map_error,
    "curvature_plots_enabled": map_curvature_plots_enabled,
    "curvature_plot_warning": map_curvature_plot_warning,
}
if map_payload is not None:
    map_summary.update({
        "n_successful_starts": map_payload["n_successful_starts"],
        "best_start_index": map_payload["best_start_index"],
        "map_log_posterior": map_payload["map_log_posterior"],
        "map_log_likelihood": map_payload["map_log_likelihood"],
        "map_log_prior": map_payload["map_log_prior"],
        "final_grad_norm": map_payload["final_grad_norm"],
        "runner_up_objective_gap": map_payload["runner_up_objective_gap"],
        "likelihood_deficiency_count": map_payload["likelihood_curvature"]["deficiency_count"],
        "likelihood_negative_direction_count": map_payload["likelihood_curvature"]["negative_direction_count"],
        "posterior_deficiency_count": map_payload["posterior_curvature"]["deficiency_count"],
        "posterior_negative_direction_count": map_payload["posterior_curvature"]["negative_direction_count"],
        "prior_rescued_parameters": map_payload["prior_rescued_parameters"],
        "boundary_parameters": map_payload["boundary_parameters"],
    })

display(summary_markdown(map_summary, "Full-Grid MAP Polish/Hessian Summary"))

if map_payload is not None:
    polish_starts_fig = starts_parallel_coords(map_payload)
    if polish_starts_fig is not None:
        display(polish_starts_fig)

if map_payload is not None and not map_curvature_plots_enabled and map_curvature_plot_warning:
    display(Markdown(f"**Curvature plot status** — {map_curvature_plot_warning}"))

if map_payload is not None and map_curvature_plots_enabled:
    map_likelihood = map_payload["likelihood_curvature"]
    map_posterior = map_payload["posterior_curvature"]
    map_interpretable = interpretable_lookup(map_likelihood["per_parameter"])

    eig_fig = eigenvalue_strip_figure(map_payload)
    if eig_fig is not None:
        display(eig_fig)

    likelihood_heatmap = family_grouped_loading_heatmap(
        parameter_names=map_likelihood.get("parameter_names") or [],
        interpretable_names=map_interpretable,
        full_vectors=map_likelihood.get("eigenvectors_normalized") or [],
        weak_direction_meta=map_likelihood["weak_directions"],
        title="Likelihood Hessian: full eigenvector matrix grouped by family",
    )
    if likelihood_heatmap is not None:
        display(likelihood_heatmap)

    likelihood_bipartite = bipartite_loading_graph(
        parameter_names=map_likelihood.get("parameter_names") or [],
        interpretable_names=map_interpretable,
        full_vectors=map_likelihood.get("eigenvectors_normalized") or [],
        weak_direction_meta=map_likelihood["weak_directions"],
        title="Likelihood Hessian: weak directions ↔ parameters",
    )
    if likelihood_bipartite is not None:
        display(likelihood_bipartite)

    ellipse_fig = laplace_pair_ellipses(
        eigvecs_norm=map_posterior.get("eigenvectors_normalized") or [],
        eigvals_norm=map_posterior["normalized_eigenvalues"],
        parameter_names=map_posterior.get("parameter_names") or [],
        interpretable_names=map_interpretable,
        weak_directions=map_posterior["weak_directions"] or map_likelihood["weak_directions"],
        title="Posterior Laplace ellipses",
    )
    if ellipse_fig is not None:
        display(Markdown(
            "**Laplace ellipse panels** — these are the 1σ/2σ contours of the Gaussian "
            "implied by the inverse posterior Hessian, projected onto the (normalized) "
            "(z-z*)/σprior plane of each weak direction's two dominant parameters. "
            "Wide ellipses = the posterior is flat in that 2D slice; the dotted axis "
            "marks the actual weak eigenvector."
        ))
        display(ellipse_fig)

    profile_fig = profile_along_directions(
        log_lik_fn=sweep_context.log_lik_fn,
        observations=map_obs,
        times=map_times,
        z_map=map_payload.get("z_map_unconstrained") or [],
        prior_std=map_payload.get("prior_std_unconstrained") or [],
        eigvecs_norm=map_likelihood.get("eigenvectors_normalized") or [],
        weak_directions=map_likelihood["weak_directions"],
        title="Likelihood Hessian honesty check",
    )
    if profile_fig is not None:
        display(Markdown(
            "**Profile vs Hessian parabola** — the blue line is the *actual* "
            "log-likelihood as we step away from the MAP along the weak eigenvector "
            "(in σ_prior · v̂ units); the orange dashed line is the *Hessian-implied* "
            "quadratic. Divergence means the Laplace approximation is misleading along "
            "that direction (the Hessian over- or under-states curvature at finite α)."
        ))
        display(profile_fig)

    bar_fig = weak_parameter_bar(
        weak_parameter_rows(map_likelihood["per_parameter"]),
        "Weak parameters from likelihood Hessian",
    )
    if bar_fig is not None:
        display(bar_fig)
